# Exploring Word Order with Bag of Words
### Authors: Isaac and Cole

In [2]:
import gensim.downloader
print(list(gensim.downloader.info()['models'].keys()))

['fasttext-wiki-news-subwords-300', 'conceptnet-numberbatch-17-06-300', 'word2vec-ruscorpora-300', 'word2vec-google-news-300', 'glove-wiki-gigaword-50', 'glove-wiki-gigaword-100', 'glove-wiki-gigaword-200', 'glove-wiki-gigaword-300', 'glove-twitter-25', 'glove-twitter-50', 'glove-twitter-100', 'glove-twitter-200', '__testing_word2vec-matrix-synopsis']


Using premade word embeddings

In [3]:
wordEmbeddings = gensim.downloader.load('glove-wiki-gigaword-300')

[==================================================] 100.0% 376.1/376.1MB downloaded


In [4]:
#undercase only
wordEmbeddings.most_similar("dog")

[('dogs', 0.7888557314872742),
 ('cat', 0.6816746592521667),
 ('pet', 0.6291598081588745),
 ('puppy', 0.593606173992157),
 ('hound', 0.5468214750289917),
 ('horse', 0.5369751453399658),
 ('animal', 0.5316445827484131),
 ('cats', 0.5080744028091431),
 ('canine', 0.5038436055183411),
 ('pets', 0.5019966959953308)]

In [7]:
import nltk
from nltk.tokenize import RegexpTokenizer
nltk.download('punkt_tab')

# regex (•ˋ _ ˊ•)
tokenizer = RegexpTokenizer(r'\w+')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Initial average method. Averaging all the words in a sentence to create a sentence vector.

In [162]:
import numpy as np

def averageSentence(sentence, wordEmbeddings):
  tokenized = tokenizer.tokenize(sentence)
  tokenized = [w.lower() for w in tokenized]

  vectors = [wordEmbeddings[w] for w in tokenized if w in wordEmbeddings.key_to_index]

  if not vectors:  # no valid tokens
    return np.zeros(wordEmbeddings.vector_size)
  return np.mean(vectors, axis=0)


Testing out the average method.

Too many stopwords...

In [163]:
wordEmbeddings.most_similar(averageSentence("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings))

[('you', 0.7987486124038696),
 ('my', 0.7915640473365784),
 ('your', 0.7680342197418213),
 ('?', 0.7425190210342407),
 ('i', 0.7415329813957214),
 ("n't", 0.7385793924331665),
 ('what', 0.7217230200767517),
 ('if', 0.7191269993782043),
 ("'ll", 0.7169417142868042),
 ('know', 0.7166122794151306)]

Get english stopwords so we can remove them

In [10]:
nltk.download('stopwords')
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


New average function. This one removes the stop words from the tokenized words before creating the vector.
Initially had w.lower(), but this led to stop words with capital letters not getting removed.

In [164]:
def averageSentenceNoStop(sentence, wordEmbeddings):
  tokenized = tokenizer.tokenize(sentence)
  tokenized = [w.lower() for w in tokenized if w not in stop_words] #needs lower() for trained embeddings

  vectors = [wordEmbeddings[w] for w in tokenized if w in wordEmbeddings.key_to_index]
  print(tokenized)
  if not vectors:  # no valid tokens
    return np.zeros(wordEmbeddings.vector_size)
  return np.mean(vectors, axis=0)

Average without stopwords

Somewhat better...


In [165]:
wordEmbeddings.most_similar(averageSentenceNoStop("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings))

['hello', 'dog', 'perchance', 'may', 'i', 'withdraw', 'wealth', 'resides', 'resplendent', 'depository']


[('you', 0.6352370977401733),
 ('?', 0.6093966364860535),
 ('your', 0.592963457107544),
 ('my', 0.591255247592926),
 ("'ll", 0.5880999565124512),
 ('i', 0.5797439217567444),
 ('if', 0.5743523836135864),
 ("n't", 0.5707759857177734),
 ('want', 0.5606643557548523),
 ('know', 0.5588496923446655)]

In [166]:
def cosine_similarity(vec1, vec2):
  if np.all(vec1 == 0) or np.all(vec2 == 0):
    return 0.0  #zero-vector case
  return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


Looking at sentences similarities.

Sentence that should be similiar is only at about 0.67.

Sentence that isn't really that similar, but not specifically so, is at 0.34.

Somewhat different sentence made of same words is considered an exact match at 1.

In [167]:
test1 = averageSentenceNoStop("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
test2 = averageSentenceNoStop("Greetings hound, I wish to take my money out of your shiny bank", wordEmbeddings)#retrieving money from the dog bank
cosine_similarity(test1, test2)

['hello', 'dog', 'perchance', 'may', 'i', 'withdraw', 'wealth', 'resides', 'resplendent', 'depository']
['greetings', 'hound', 'i', 'wish', 'take', 'money', 'shiny', 'bank']


0.6692991

In [168]:
test1 = averageSentenceNoStop("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings)#retrieving money from the dog bank
test2 = averageSentenceNoStop("Oh no, watch out for Dracula and his ivory tongs of pain and gloom!", wordEmbeddings)#getting attacked by harrowing creature
cosine_similarity(test1, test2)

['hello', 'dog', 'perchance', 'may', 'i', 'withdraw', 'wealth', 'resides', 'resplendent', 'depository']
['oh', 'watch', 'dracula', 'ivory', 'tongs', 'pain', 'gloom']


0.3434914

In [169]:
test1 = averageSentenceNoStop("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings)#retrieving money from the dog bank
test2 = averageSentenceNoStop("I perchance your dog in my wealth depository, withdraw there may resides that Hello resplendent!", wordEmbeddings)#there is a dog in my vault, do not let the devil shine!
cosine_similarity(test1, test2)

['hello', 'dog', 'perchance', 'may', 'i', 'withdraw', 'wealth', 'resides', 'resplendent', 'depository']
['i', 'perchance', 'dog', 'wealth', 'depository', 'withdraw', 'may', 'resides', 'hello', 'resplendent']


1.0

Try to account for word order now
Also originally used w.lower(), same as above where it was missing certain stopwords with capital letters.

In [170]:
def averageOrder(sentence, model):
  tokenized = tokenizer.tokenize(sentence)
  tokenized = [w.lower() for w in tokenized if w not in stop_words]
  vectors = [wordEmbeddings[w] for w in tokenized if w in wordEmbeddings.key_to_index]

  #add vectors between sequential words to overall all vector
  for i in range(len(tokenized) - 1):
    if tokenized[i] in model.key_to_index and tokenized[i+1] in model.key_to_index:
      diff = model[tokenized[i+1]] - model[tokenized[i]]
      vectors.append(diff)

  if not vectors:  # no valid tokens
    return np.zeros(wordEmbeddings.vector_size)
  return np.mean(vectors, axis=0)

Some more solid words as our most similar. A differently ordered sentence also returns different similar words. Some words also appear to a have stronger similarity compared to the simple average method.

In [171]:
wordEmbeddings.most_similar(averageOrder("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings))

[('depository', 0.5803999304771423),
 ('you', 0.49588543176651),
 ('your', 0.48407721519470215),
 ('if', 0.47585180401802063),
 ("n't", 0.46870097517967224),
 ('must', 0.4638955295085907),
 ('?', 0.4596001207828522),
 ('my', 0.4592358469963074),
 ('not', 0.45809686183929443),
 ('does', 0.45635759830474854)]

In [172]:
wordEmbeddings.most_similar(averageOrder("I perchance your dog in my wealth depository, withdraw there may resides that Hello resplendent!", wordEmbeddings))

[('your', 0.433493047952652),
 ('resplendent', 0.42802557349205017),
 ('yours', 0.40732279419898987),
 ('dog', 0.3979795277118683),
 ('?', 0.39035940170288086),
 ('eyes', 0.3728010356426239),
 ('you', 0.37174898386001587),
 ('my', 0.37113046646118164),
 ('somewhere', 0.36731988191604614),
 ('cat', 0.36422887444496155)]

The cosine similarities for the sentences when accounting for order.
The similar sentences drop from 0.6 to 0.5, not great.
The not similar sentences drop from 0.34 to 0.16, this is improvement because these shouldn't be seen as similar.
Then we see that the sentence with the words changed around was originally 1, but is now 0.86 when accounting for word order. This is good because we don't want these sentences to be seen as the same, but they still have similar words.

In [173]:
test1 = averageOrder("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
test2 = averageOrder("Greetings hound, I wish to take my money out of your shiny bank", wordEmbeddings)#retrieving money from the dog bank
cosine_similarity(test1, test2)

0.55976063

In [174]:
test1 = averageOrder("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings)#retrieving money from the dog bank
test2 = averageOrder("Oh no, watch out for Dracula and his ivory tongs of pain and gloom!", wordEmbeddings)#getting attacked by harrowing creature
cosine_similarity(test1, test2)

0.16630912

In [175]:
test1 = averageOrder("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings)#retrieving money from the dog bank
test2 = averageOrder("I perchance your dog in my wealth depository, withdraw there may resides that Hello resplendent ", wordEmbeddings)#there is a dog in my vault, do not let the devil shine!
cosine_similarity(test1, test2)

0.8618673

This last one has 2 of the exact same sentences. We should get 1 and we do.

In [176]:
test1 = averageOrder("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
test2 = averageOrder("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
cosine_similarity(test1, test2)

1.0

Now lets ignore the individual word embeddings and only use the averaged vectors connecting the words.

In [188]:
def averageOrderOnly(sentence, model):
  tokenized = tokenizer.tokenize(sentence)
  tokenized = [w.lower() for w in tokenized if w not in stop_words]
  vectors = []

  #add vectors between sequential words to overall all vector
  for i in range(len(tokenized) - 1):
    if tokenized[i] in model.key_to_index and tokenized[i+1] in model.key_to_index:
      diff = model[tokenized[i+1]] - model[tokenized[i]]
      vectors.append(diff)

  if not vectors:  # no valid tokens
    return np.zeros(wordEmbeddings.vector_size)
  return np.mean(vectors, axis=0)

In [178]:
wordEmbeddings.most_similar(averageOrderOnly("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings))

[('depository', 0.8044969439506531),
 ('depositary', 0.5936049818992615),
 ('adrs', 0.4812626540660858),
 ('receipts', 0.43246087431907654),
 ('gdrs', 0.4004581868648529),
 ('adr', 0.37954673171043396),
 ('bb94', 0.36855506896972656),
 ('receipt', 0.3427126109600067),
 ('depositories', 0.32117658853530884),
 ('b.a.t', 0.315734326839447)]

In [179]:
wordEmbeddings.most_similar(averageOrderOnly("I perchance your dog in my wealth depository, withdraw there may resides that Hello resplendent!", wordEmbeddings))

[('resplendent', 0.6658350229263306),
 ('cw96', 0.4920865595340729),
 ('brett.clanton@chron.com', 0.48407110571861267),
 ('3.6730', 0.4826506972312927),
 ('tom.fowler@chron.com', 0.477710098028183),
 ('ooooooooooooooooooooooooooooooooooooooo', 0.4705314636230469),
 ('el1l', 0.46415281295776367),
 ('25aou94', 0.4640451669692993),
 ('viewport', 0.4606526792049408),
 ('___________________________________________________________',
  0.45650768280029297)]

I'm not sure what's going on here, maybe we stumbled upon cluster of stinky tokens from our big wikipedia data when forming these sentence embeddings...

In [183]:
wordEmbeddings.most_similar("25aou94")

[('23aou94', 0.8609238862991333),
 ('cw96', 0.7666194438934326),
 ('mangxamba', 0.6813809871673584),
 ('k586-1', 0.6677289009094238),
 ('___________________________________________________________',
  0.6577410697937012),
 ('kd96', 0.6477249264717102),
 ('thongrung', 0.6408474445343018),
 ('rosnazura', 0.6361333727836609),
 ('rw97', 0.6307600140571594),
 ('tom.fowler@chron.com', 0.6303925514221191)]

While the scores are all fairly low, it appears that our sentence that should be more similar has a higher score than the shuffled sentence.

In [189]:
test1 = averageOrderOnly("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
test2 = averageOrderOnly("Greetings hound, I wish to take my money out of your shiny bank", wordEmbeddings)#retrieving money from the dog bank
cosine_similarity(test1, test2)

0.2304637

In [190]:
test1 = averageOrderOnly("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings)#retrieving money from the dog bank
test2 = averageOrderOnly("Oh no, watch out for Dracula and his ivory tongs of pain and gloom!", wordEmbeddings)#getting attacked by harrowing creature
cosine_similarity(test1, test2)

0.18755513

In [199]:
test1 = averageOrderOnly("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings)#retrieving money from the dog bank
test2 = averageOrderOnly("I perchance your dog in my wealth depository, withdraw there may resides that Hello resplendent ", wordEmbeddings)#there is a dog in my vault, do not let the devil shine!
cosine_similarity(test1, test2)

0.16643783

In [192]:
test1 = averageOrderOnly("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
test2 = averageOrderOnly("Hello there dog, perchance may I withdraw my wealth that resides in your resplendent depository!", wordEmbeddings) #retrieving money from the dog bank
cosine_similarity(test1, test2)

1.0000001

In contemplation, any method averaging individual word embeddings will likely be biased towards a shuffled sentence compared to any other sentence regardless of similarity. Our sequential vector method kind of sniffs out the difference here, but the overall score seems too low for the actual sentence representation to be great.